# Image support

Claude accepts images inside a user message as an **image content block**.
The block carries the raw bytes base64-encoded; nothing is uploaded ahead of time,
the data travels inline with the request.
Up to 100 images per request, charged by pixel area.

## Content block shape

```python
{
    "type": "image",
    "source": {"type": "base64", "media_type": "image/png", "data": "<base64>"},
}
```

`attach_file(path)` from `files_api.py` builds this for you from a local file, auto-detecting the MIME type.
The same helper is reused by `3_pdf` and `4_citations` with a `document` block shape when the file is a PDF.

## Why strong prompting matters

Image accuracy depends almost entirely on prompt structure.
A one-liner like "what's in this image?" produces vague answers.
The demo uses a structured 5-step prompt
(identify residence, assess tree overhang, score fire risk, etc.)
that pins Claude to a specific analysis.

Model: `claude-sonnet-4-6`.


In [1]:
from chat_tools import add_user_message, chat, text_from_message
from files_api import attach_file
from models import MODEL_SONNET

In [2]:
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a \"bridge\" between wildland vegetation and the structure

4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [3]:
messages = []
image_block = attach_file("images/prop7.png")
text_block = {"type": "text", "text": prompt}
add_user_message(messages, [image_block, text_block])

response = chat(messages, model=MODEL_SONNET, max_tokens=4000)
print(text_from_message(response))

## Satellite Image Fire Risk Analysis

**1. Residence Identification:**
The primary residence is a large, multi-section structure with a gray/tan roof visible in the center-lower portion of the image, featuring an irregular L-shaped or multi-wing footprint with what appears to be a driveway or cleared area on its northwest side, surrounded on all sides by dense woodland with minimal setback from the tree line.

**2. Tree Overhang Analysis:**
Multiple mature trees have canopies that extend directly over portions of the roof, particularly on the western, southern, and eastern sides of the structure, with an estimated **50–75% of the roof area** falling beneath or immediately adjacent to overhanging tree canopies, with notably dense overhang visible along the left (west) and lower (south) roof sections.

**3. Fire Risk Assessment:**
The dense overhanging branches create multiple ember catch points along the roof valleys and edges, and the continuous vegetation-to-canopy-to-roof pathway re